# Building a Local AI Coding Assistant: Ollama, Continue, and Agentic Tools

## 1. Objective

This guide explains how to set up **Ollama** to run open-source models locally and connect them to VS Code for both **chat** and **agentic coding** using extensions like **Continue**, **Cline** (formerly Claude Dev), **OpenHands**, and **Claude Code**.

Unlike `llama.cpp` (which requires manual builds and configuration), Ollama provides a Docker-like experience for LLMs. It abstracts away the complexity of hardware backends and provides a unified REST API.

---

## 2. Installing Ollama on Windows

Ollama natively supports Windows and handles GPU acceleration automatically.

1. Download the Windows installer from [ollama.com](https://ollama.com/download/windows).
2. Run the installer.
3. Open a **new terminal** (Command Prompt or PowerShell) and verify the installation:

In [ ]:
!   ollama --version


Ollama runs as a background service on `[http://127.0.0.1:11434`.](http://127.0.0.1:11434`.)

---

## 3. Pulling and Running Open Source Models

For coding and agentic workflows, you need models that are strong at reasoning, tool-use, and code generation.

### Recommended Models for Coding
- **Qwen 2.5 Coder (7B - 32B)**: Currently the state-of-the-art for local coding.
- **Llama 3.1 (8B)**: Excellent general reasoning and chat.
- **DeepSeek Coder V2**: Very strong alternative for complex reasoning.

### How to pull and test a model

To pull and run Qwen 2.5 Coder (7B parameters):

In [ ]:
!ollama run qwen2.5-coder:7b

*Note: If the model isn't downloaded yet, Ollama will download it automatically. This requires about 4-5 GB of disk space.*

Once downloaded, you will drop into an interactive chat prompt. Type `/bye` to exit. The Ollama service remains running in the background.

---

## 4. Chat Mode vs. Agentic Mode

When using AI for coding, there are two main interaction paradigms:

1. **Chat / Autocomplete Mode (e.g., Continue)**
   - **Behavior**: You highlight code, ask questions, or the AI suggests completions as you type.
   - **Requirement**: Fast response times, medium context windows (8k).
   - **Best Models**: `qwen2.5-coder:7b` or `qwen2.5-coder:1.5b` (for lightning-fast autocomplete).

2. **Agentic Mode (e.g., Cline, OpenHands, Claude Code)**
   - **Behavior**: The AI operates as an autonomous agent. It can read your file system, execute terminal commands, edit files, and plan multi-step tasks.
   - **Requirement**: High reasoning capabilities, strong JSON/Tool-calling support, large context windows (32k+).
   - **Best Models**: `qwen2.5-coder:32b` (if you have 24GB+ RAM) or `llama3.1:8b`.

---

## 5. Setting Up "Continue" for Chat and Autocomplete

[Continue](https://www.continue.dev/) is a VS Code extension that provides side-panel chat and inline autocomplete.

### Installation
1. Install the **Continue** extension from the VS Code Marketplace.
2. Open the Continue side panel and click the Gear icon to open `config.json`.

### Configuration for Ollama
Update your `config.json` to point to your local Ollama instance:

```json
{
  "models": [
    {
      "title": "Ollama Qwen 2.5 Coder",
      "provider": "ollama",
      "model": "qwen2.5-coder:7b",
      "apiBase": "http://127.0.0.1:11434"
    }
  ],
  "tabAutocompleteModel": {
    "title": "Ollama Autocomplete",
    "provider": "ollama",
    "model": "qwen2.5-coder:1.5b",
    "apiBase": "http://127.0.0.1:11434"
  }
}
```
*Tip: We use a smaller 1.5B model for autocomplete so it generates code instantly without lag, while using the 7B model for heavier chat reasoning.*

---

## 6. Setting Up Agentic Tools (Cline / Claude Code / OpenHands)

**🔥 Latest Update:** Recent versions of Ollama have introduced **100% native OpenAI API compatibility** and **native Tool Calling**. This is a massive game-changer. You no longer need middleware wrappers, community forks, or tricky configuration. Any tool that supports OpenAI can now treat your local Ollama instance exactly as if it were the OpenAI API.

### Using Cline (VS Code Extension)
Cline is an autonomous agent inside VS Code that can create and edit files.

1. Install **Cline** from the VS Code Marketplace.
2. In the Cline settings panel, look for the **API Provider** dropdown.
3. Select **OpenAI Compatible** (or the dedicated Ollama provider if available).
4. Enter the Base URL: `[http://127.0.0.1:11434/v1`](http://127.0.0.1:11434/v1`)
5. Enter the API Key: `ollama` (required but unused by Ollama)
6. Type the exact Model ID you pulled earlier (e.g., `qwen2.5-coder:32b`).

*Note: Since Ollama handles tool calling natively, you can now seamlessly use models like Qwen 2.5 and Llama 3.1 for complex multi-step edits without hallucinating tool calls.*

### Using Claude Code (CLI) or OpenAI-compatible CLI tools
Claude Code is a terminal-based agent. Thanks to Ollama's OpenAI drop-in compatibility, you can route terminal agents directly to your local models simply by setting two environment variables.

In PowerShell:

In [ ]:
!$env:OPENAI_API_KEY="ollama"
!$env:OPENAI_BASE_URL="http://127.0.0.1:11434/v1"


In Bash:

In [ ]:
!export OPENAI_API_KEY="ollama"
!export OPENAI_BASE_URL="http://127.0.0.1:11434/v1"

After setting these, simply launch your agent tool and it will execute entirely locally using Ollama's built-in tool calling support!

### Using OpenHands (formerly OpenDevin)
OpenHands is a powerful, sandboxed autonomous AI software engineer.

1. OpenHands runs via Docker. Start it with Ollama configured:

In [ ]:
!   docker run -it --pull=always \
!       -e SANDBOX_USER_ID=$(id -u) \
!       -e WORKSPACE_MOUNT_PATH=$(pwd)/workspace \
!       -e LLM_API_KEY="ollama" \
!       -e LLM_BASE_URL="http://host.docker.internal:11434/v1" \
!       -e LLM_MODEL="openai/qwen2.5-coder:7b" \
!       -v $(pwd)/workspace:/opt/workspace_base \
!       -v /var/run/docker.sock:/var/run/docker.sock \
!       -p 3000:3000 \
!       docker.all-hands.dev/all-hands-ai/openhands:0.9

2. Note the use of `[http://host.docker.internal:11434/v1`](http://host.docker.internal:11434/v1`) — this utilizes Ollama's native OpenAI endpoint directly from the Docker container.

---

## 7. Troubleshooting

| Problem | Solution |
|---|---|
| Ollama says "connection refused" | Ensure the Ollama app is running in the Windows system tray. |
| Models are too slow | Ensure your monitor is plugged into your dedicated GPU. In Windows Task Manager, check if the GPU is being utilized. If VRAM is maxed out, use a smaller model (e.g., 7B instead of 32B). |
| Agentic tools hallucinate tool calls | Agentic behavior requires strong instruction following. Upgrade to a larger model (32B+), or use a model specifically fine-tuned for tool calling. |

---

## 8. Summary of Workflow

1. Start Ollama (`ollama run qwen2.5-coder:7b`).
2. Open VS Code.
3. For Chat/Autocomplete: Use **Continue** pointing to `localhost:11434`.
4. For Agentic Tasks: Use **Cline** or **OpenHands** pointing to `localhost:11434`.
